<a href="https://colab.research.google.com/github/anhndt0310-jpg/Cyberbullying-Detection-MultiClass/blob/main/vietnamese_cyberbullying_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cài đặt các thư viện cần thiết cho bài toán
!pip install transformers torch pandas numpy scikit-learn

# Kết nối với Google Drive để lấy file
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra xem GPU có hoạt động không
import torch
print("GPU đang sẵn sàng:", torch.cuda.is_available())

Mounted at /content/drive
GPU đang sẵn sàng: True


In [6]:
import pandas as pd

import pandas as pd

# Đường dẫn đến file (Nếu file của bạn có tên khác, hãy sửa lại phần trong ngoặc)
# Giả sử file của bạn nằm ngay ngoài thư mục gốc của My Drive
train_path = '/content/drive/MyDrive/df_train_clean.csv'
test_path = '/content/drive/MyDrive/df_test_clean.csv'

# Đọc dữ liệu
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# In ra thông tin để xác nhận dữ liệu đã được nạp
print("Số dòng trong tập Train:", len(train_df))
print("Các cột có trong file:", train_df.columns.tolist())

# Hiển thị 3 dòng đầu tiên để bạn kiểm tra tên cột
print("\n--- 3 dòng đầu của tập Train ---")
print(train_df.head(3))

Số dòng trong tập Train: 7000
Các cột có trong file: ['Unnamed: 0', 'content', 'Constructiveness', 'Toxicity', 'Title', 'Topic', 'content_clean']

--- 3 dòng đầu của tập Train ---
   Unnamed: 0                                            content  \
0        6326                               Thật tuyệt vời...!!!   
1        7835  mỹ đã tuột dốc quá nhiều rồi, giờ muốn vực dậy...   
2        4690  tôi thấy người lái xe hơi bấm còi mới là người...   

   Constructiveness  Toxicity  \
0                 0         0   
1                 1         0   
2                 1         1   

                                               Title     Topic  \
0  Những 'bước tiến diệu kỳ' của Trúc Nhi - Diệu Nhi   SucKhoe   
1  Hình tượng Mỹ sụp đổ trong lòng người dân thế ...   TheGioi   
2         Cả trăm người đạp xe thể dục bịt kín đường  OtoXemay   

                                       content_clean  
0                               Thật tuyệt vời...!!!  
1  mỹ đã tuột dốc quá nhiều rồi, giờ mu

In [4]:
# Trích xuất cho tập Train
print("--- Đang trích xuất đặc trưng cho tập Train (7000 câu) ---")
X_train_emb = get_phobert_embeddings(train_df['content_clean'].fillna('').tolist())
y_train = train_df['Toxicity'].values

# Trích xuất cho tập Test
# (Giả sử bạn đã nạp biến test_df từ các bước trước)
print("\n--- Đang trích xuất đặc trưng cho tập Test ---")
X_test_emb = get_phobert_embeddings(test_df['content_clean'].fillna('').tolist())
y_test = test_df['Toxicity'].values

print("\nHoàn tất! Dữ liệu đã sẵn sàng để huấn luyện.")

--- Đang trích xuất đặc trưng cho tập Train (7000 câu) ---


100%|██████████| 438/438 [00:37<00:00, 11.59it/s]



--- Đang trích xuất đặc trưng cho tập Test ---


100%|██████████| 63/63 [00:05<00:00, 11.98it/s]


Hoàn tất! Dữ liệu đã sẵn sàng để huấn luyện.


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# 1. Khởi tạo mô hình
# Cần dùng class_weight='balanced' để mô hình không bị "lười"
model = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)

# 2. Huấn luyện mô hình
print("Đang huấn luyện Logistic Regression trên vector của PhoBERT...")
model.fit(X_train_emb, y_train)

# 3. Dự đoán trên tập Test
y_pred = model.predict(X_test_emb)

# 4. Xuất kết quả
print("\n=== KẾT QUẢ CUỐI CÙNG (PHOBERT + LOGISTIC REGRESSION) ===")
print(classification_report(y_test, y_pred))

print("\n=== MA TRẬN NHẦM LẪN ===")
print(confusion_matrix(y_test, y_pred))

Đang huấn luyện Logistic Regression trên vector của PhoBERT...

=== KẾT QUẢ CUỐI CÙNG (PHOBERT + LOGISTIC REGRESSION) ===
              precision    recall  f1-score   support

           0       0.94      0.81      0.87       890
           1       0.28      0.61      0.39       110

    accuracy                           0.79      1000
   macro avg       0.61      0.71      0.63      1000
weighted avg       0.87      0.79      0.82      1000


=== MA TRẬN NHẦM LẪN ===
[[719 171]
 [ 43  67]]
